In [ ]:
# ============================================================
# VQC on German Credit Dataset (credit-g, OpenML)
# - 6 qubits (PCA to 6 components, original features > 6)
# - ZZFeatureMap as data encoding (QSVM-style)
# - 5 different TwoLocal ansatz configurations
# - EstimatorQNN + TorchConnector + PyTorch training
# ============================================================

# requirements are commented below for reference 
# !pip install -q qiskit qiskit-aer qiskit-machine-learning torch scikit-learn pandas

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim

from qiskit.circuit.library import ZZFeatureMap, TwoLocal
from qiskit.quantum_info import Pauli
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

# -------------------------
# Global config
# -------------------------
RANDOM_SEED = 42
NUM_QUBITS = 6                   # qubits for VQC
PCA_COMPONENTS = NUM_QUBITS      # PCA -> 6 components
TRAIN_SIZE_FRACTION = 0.7
MAX_TRAIN_SAMPLES = 700          # cap training set to keep runtime reasonable
EPOCHS = 7                       # training epochs per ansatz (adjust if slow)
BATCH_SIZE = 32
LEARNING_RATE = 0.01

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.set_default_dtype(torch.float32)

# ============================================================
# 1. Load German Credit (credit-g) from OpenML
# ============================================================
print("Fetching German Credit dataset from OpenML...")
data = fetch_openml("credit-g", version=1, as_frame=True)
df = data.frame
print("Raw dataframe shape:", df.shape)
print("Columns:", df.columns.tolist())

# Target column is 'class' with values 'good'/'bad'
y_raw = df["class"].apply(lambda v: 1 if v == "good" else 0)  # 1 = good, 0 = bad
X_raw = df.drop(columns=["class"])

print("\nClass distribution (full):")
print(y_raw.value_counts())

# ============================================================
# 2. One-hot encode categoricals + scale + PCA -> 6 comps
# ============================================================
# One-hot encode all categorical columns
X_encoded = pd.get_dummies(X_raw, drop_first=True)
print("\nAfter one-hot encoding, shape:", X_encoded.shape)

# Train/test split (on encoded features)
X_train_enc, X_test_enc, y_train, y_test = train_test_split(
    X_encoded.values,
    y_raw.values,
    train_size=TRAIN_SIZE_FRACTION,
    random_state=RANDOM_SEED,
    stratify=y_raw.values
)

print("Initial split sizes: train =", X_train_enc.shape[0], ", test =", X_test_enc.shape[0])

# Optionally cap training size to keep runtime reasonable
if X_train_enc.shape[0] > MAX_TRAIN_SAMPLES:
    idx = np.random.RandomState(RANDOM_SEED).choice(
        X_train_enc.shape[0], size=MAX_TRAIN_SAMPLES, replace=False
    )
    X_train_enc = X_train_enc[idx]
    y_train = y_train[idx]
    print("Capped training set to", X_train_enc.shape[0], "samples")

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_test_scaled  = scaler.transform(X_test_enc)

# PCA to 6 components (qubits < original encoded features)
pca = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_SEED)
X_train_pca = pca.fit_transform(X_train_scaled).astype(np.float32)
X_test_pca  = pca.transform(X_test_scaled).astype(np.float32)

print("\nAfter PCA, train shape:", X_train_pca.shape, "test shape:", X_test_pca.shape)
print("Original encoded feature dim:", X_encoded.shape[1], 
      " > qubits:", NUM_QUBITS, "OK")

# ============================================================
# 3. Build ZZFeatureMap (QSVM-style feature encoding)
# ============================================================
feature_map = ZZFeatureMap(
    feature_dimension=NUM_QUBITS,
    reps=1,
    entanglement="linear"
)

print("\nFeature map created: ZZFeatureMap with",
      NUM_QUBITS, "qubits and reps=1.")

# ============================================================
# 4. Define 5 different TwoLocal ansatz configurations
# ============================================================
ansatz_configs = [
    # {
    #     "name": "ry_linear_reps1",
    #     "rotation_blocks": "ry",
    #     "entanglement_blocks": "cx",
    #     "entanglement": "linear",
    #     "reps": 1,
    # },
    {
        "name": "ry_linear_reps2",
        "rotation_blocks": "ry",
        "entanglement_blocks": "cx",
        "entanglement": "linear",
        "reps": 2,
    },
    {
        "name": "ry_full_reps1",
        "rotation_blocks": "ry",
        "entanglement_blocks": "cx",
        "entanglement": "full",
        "reps": 1,
    },
    {
        "name": "ryrz_linear_reps1",
        "rotation_blocks": ["ry", "rz"],
        "entanglement_blocks": "cx",
        "entanglement": "linear",
        "reps": 1,
    },
    {
        "name": "ryrz_full_reps2",
        "rotation_blocks": ["ry", "rz"],
        "entanglement_blocks": "cx",
        "entanglement": "full",
        "reps": 2,        # most expressive / heaviest
    },
]

print("\nAnsatz configurations:")
for cfg in ansatz_configs:
    print("  -", cfg["name"], ":", cfg)

# ============================================================
# 5. Quantum backend + QNN wrapper
# ============================================================
estimator = AerEstimator()
observable = Pauli("Z" + "I" * (NUM_QUBITS - 1))  # measure Z on first qubit


def build_ansatz(cfg):
    return TwoLocal(
        num_qubits=NUM_QUBITS,
        rotation_blocks=cfg["rotation_blocks"],
        entanglement_blocks=cfg["entanglement_blocks"],
        entanglement=cfg["entanglement"],
        reps=cfg["reps"],
    )


class VQCModel(nn.Module):
    """
    Wrap an EstimatorQNN in a PyTorch model:
    - input: PCA features (6)
    - QNN: expectation value in [-1, 1]
    - Linear layer + Sigmoid -> probability of class 1
    """
    def __init__(self, qnn):
        super().__init__()
        self.qnn = TorchConnector(qnn)
        self.fc = nn.Linear(1, 1)

    def forward(self, x):
        # x: (batch, 6)
        exp_val = self.qnn(x)        # (batch, 1)
        logits  = self.fc(exp_val)   # (batch, 1)
        return torch.sigmoid(logits)


def train_vqc(cfg, X_train, y_train, X_test, y_test,
              epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE):
    print("\n" + "="*80)
    print("Training VQC with ansatz:", cfg["name"])
    print("="*80)

    ansatz = build_ansatz(cfg)
    vqc_circuit = feature_map.compose(ansatz)

    input_params  = list(feature_map.parameters)
    weight_params = list(ansatz.parameters)

    qnn = EstimatorQNN(
        estimator=estimator,
        circuit=vqc_circuit,
        input_params=input_params,
        weight_params=weight_params,
        observables=observable,
        input_gradients=True,
    )

    model = VQCModel(qnn)

    X_tr_t = torch.tensor(X_train, dtype=torch.float32)
    y_tr_t = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    X_te_t = torch.tensor(X_test, dtype=torch.float32)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()

    for ep in range(epochs):
        # shuffle
        perm = torch.randperm(len(X_tr_t))
        X_tr_t = X_tr_t[perm]
        y_tr_t = y_tr_t[perm]

        total_loss = 0.0
        model.train()
        for i in range(0, len(X_tr_t), batch_size):
            xb = X_tr_t[i:i+batch_size]
            yb = y_tr_t[i:i+batch_size]

            optimizer.zero_grad()
            preds = model(xb)
            loss  = loss_fn(preds, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {ep+1}/{epochs} - loss: {total_loss:.4f}")

    # Evaluation
    model.eval()
    with torch.no_grad():
        probs = model(X_te_t).numpy().flatten()
    y_pred = (probs > 0.5).astype(int)

    acc = accuracy_score(y_test, y_pred)
    print("\nTest Accuracy for", cfg["name"], ":", acc)
    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, digits=4))

    return acc, model, y_pred


# ============================================================
# 6. Run all 5 ansatz configurations & compare
# ============================================================
results = []
models = {}

for cfg in ansatz_configs:
    acc, mdl, y_pred = train_vqc(cfg, X_train_pca, y_train, X_test_pca, y_test)
    results.append((cfg["name"], acc))
    models[cfg["name"]] = mdl

# Sort by accuracy
results_sorted = sorted(results, key=lambda x: x[1], reverse=True)

print("\n" + "#"*80)
print("FINAL SUMMARY: VQC on German Credit (6 qubits)")
print("#"*80)
for name, acc in results_sorted:
    print(f"{name:20s} -> Accuracy: {acc:.4f}")
print("#"*80)

print("\nDone.")


Fetching German Credit dataset from OpenML...
Raw dataframe shape: (1000, 21)
Columns: ['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings_status', 'employment', 'installment_commitment', 'personal_status', 'other_parties', 'residence_since', 'property_magnitude', 'age', 'other_payment_plans', 'housing', 'existing_credits', 'job', 'num_dependents', 'own_telephone', 'foreign_worker', 'class']

Class distribution (full):
class
1    700
0    300
Name: count, dtype: int64

After one-hot encoding, shape: (1000, 48)
Initial split sizes: train = 700 , test = 300

After PCA, train shape: (700, 6) test shape: (300, 6)
Original encoded feature dim: 48  > qubits: 6 OK

Feature map created: ZZFeatureMap with 6 qubits and reps=1.

Ansatz configurations:
  - ry_linear_reps2 : {'name': 'ry_linear_reps2', 'rotation_blocks': 'ry', 'entanglement_blocks': 'cx', 'entanglement': 'linear', 'reps': 2}
  - ry_full_reps1 : {'name': 'ry_full_reps1', 'rotation_blocks': 'ry', 'e

C:\Users\hisham.fawad\AppData\Local\Temp\ipykernel_38040\570320965.py:209: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  qnn = EstimatorQNN(


In [1]:
# ============================================================
# VQC on German Credit Dataset (credit-g, OpenML)
# - 6 qubits (PCA to 6 components, original features > 6)
# - ZZFeatureMap as data encoding (QSVM-style)
# - 3 different TwoLocal ansatz configurations
# - EstimatorQNN + TorchConnector + PyTorch training
# - Features scaled to [-pi, pi]
# ============================================================

# If running in Colab, uncomment:
# !pip install -q qiskit qiskit-aer qiskit-machine-learning torch scikit-learn pandas

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim

from qiskit.circuit.library import ZZFeatureMap, TwoLocal
from qiskit.quantum_info import Pauli
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

# -------------------------
# Global config
# -------------------------
RANDOM_SEED = 42
NUM_QUBITS = 6                   # qubits for VQC
PCA_COMPONENTS = NUM_QUBITS      # PCA -> 6 components
TRAIN_SIZE_FRACTION = 0.7

# Make it fast:
MAX_TRAIN_SAMPLES = 200          # cap training set to keep runtime reasonable
EPOCHS = 3                       # epochs per ansatz
BATCH_SIZE = 16
LEARNING_RATE = 0.01

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.set_default_dtype(torch.float32)

# ============================================================
# 1. Load German Credit (credit-g) from OpenML
# ============================================================
print("Fetching German Credit dataset from OpenML...")
data = fetch_openml("credit-g", version=1, as_frame=True)
df = data.frame
print("Raw dataframe shape:", df.shape)
print("Columns:", df.columns.tolist())

# Target column is 'class' with values 'good'/'bad'
y_raw = df["class"].apply(lambda v: 1 if v == "good" else 0)  # 1 = good, 0 = bad
X_raw = df.drop(columns=["class"])

print("\nClass distribution (full):")
print(y_raw.value_counts())

# ============================================================
# 2. One-hot encode categoricals
# ============================================================
X_encoded = pd.get_dummies(X_raw, drop_first=True)
print("\nAfter one-hot encoding, shape:", X_encoded.shape)

# Train/test split (on encoded features)
X_train_enc, X_test_enc, y_train, y_test = train_test_split(
    X_encoded.values,
    y_raw.values,
    train_size=TRAIN_SIZE_FRACTION,
    random_state=RANDOM_SEED,
    stratify=y_raw.values
)

print("Initial split sizes: train =", X_train_enc.shape[0], ", test =", X_test_enc.shape[0])

# Optionally cap training size to keep runtime reasonable
if X_train_enc.shape[0] > MAX_TRAIN_SAMPLES:
    rng = np.random.RandomState(RANDOM_SEED)
    idx = rng.choice(X_train_enc.shape[0], size=MAX_TRAIN_SAMPLES, replace=False)
    X_train_enc = X_train_enc[idx]
    y_train = y_train[idx]
    print("Capped training set to", X_train_enc.shape[0], "samples")

# ============================================================
# 3. Standardize + PCA + scale to [-pi, pi]
# ============================================================
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train_enc)
X_test_std  = scaler.transform(X_test_enc)

# PCA to 6 components (qubits < original encoded features)
pca = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_SEED)
X_train_pca = pca.fit_transform(X_train_std).astype(np.float32)
X_test_pca  = pca.transform(X_test_std).astype(np.float32)

print("\nAfter PCA, train shape:", X_train_pca.shape, "test shape:", X_test_pca.shape)
print("Original encoded feature dim:", X_encoded.shape[1], 
      " > qubits:", NUM_QUBITS, "OK")

# Scale PCA features to [-pi, pi]
pi = np.pi
max_abs = np.max(np.abs(X_train_pca))
if max_abs < 1e-8:
    X_train_scaled = np.zeros_like(X_train_pca)
    X_test_scaled  = np.zeros_like(X_test_pca)
    print("\nWarning: PCA components are ~0, using zeros.")
else:
    X_train_scaled = (X_train_pca / max_abs) * pi
    X_test_scaled  = (X_test_pca / max_abs) * pi

print("\nScaled PCA feature range (train):",
      X_train_scaled.min(), "to", X_train_scaled.max(), " (approx [-pi, pi])")

# ============================================================
# 4. Build ZZFeatureMap (QSVM-style feature encoding)
# ============================================================
feature_map = ZZFeatureMap(
    feature_dimension=NUM_QUBITS,
    reps=1,
    entanglement="linear"
)

print("\nFeature map: ZZFeatureMap with",
      NUM_QUBITS, "qubits and reps=1.")

# ============================================================
# 5. Define 3 TwoLocal ansatz configurations
# ============================================================
ansatz_configs = [
    {
        "name": "ry_linear_reps1",
        "rotation_blocks": "ry",
        "entanglement_blocks": "cx",
        "entanglement": "linear",
        "reps": 1,
    },
    {
        "name": "ry_linear_reps2",
        "rotation_blocks": "ry",
        "entanglement_blocks": "cx",
        "entanglement": "linear",
        "reps": 2,
    },
    {
        "name": "ry_full_reps1",
        "rotation_blocks": "ry",
        "entanglement_blocks": "cx",
        "entanglement": "full",
        "reps": 1,
    },
]

print("\nAnsatz configurations:")
for cfg in ansatz_configs:
    print("  -", cfg["name"], ":", cfg)

# ============================================================
# 6. Quantum backend + QNN wrapper
# ============================================================
estimator = AerEstimator()
observable = Pauli("Z" + "I" * (NUM_QUBITS - 1))  # measure Z on first qubit


def build_ansatz(cfg):
    return TwoLocal(
        num_qubits=NUM_QUBITS,
        rotation_blocks=cfg["rotation_blocks"],
        entanglement_blocks=cfg["entanglement_blocks"],
        entanglement=cfg["entanglement"],
        reps=cfg["reps"],
    )


class VQCModel(nn.Module):
    """
    Wrap an EstimatorQNN in a PyTorch model:
    - input: 6 scaled PCA features in [-pi, pi]
    - QNN: expectation value in [-1, 1]
    - Linear layer + Sigmoid -> probability of class 1
    """
    def __init__(self, qnn):
        super().__init__()
        self.qnn = TorchConnector(qnn)
        self.fc = nn.Linear(1, 1)

    def forward(self, x):
        # x: (batch, 6)
        exp_val = self.qnn(x)        # (batch, 1)
        logits  = self.fc(exp_val)   # (batch, 1)
        return torch.sigmoid(logits)


def train_vqc(cfg, X_train, y_train, X_test, y_test,
              epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE):
    print("\n" + "="*80)
    print("Training VQC with ansatz:", cfg["name"])
    print("="*80)

    ansatz = build_ansatz(cfg)
    vqc_circuit = feature_map.compose(ansatz)

    input_params  = list(feature_map.parameters)
    weight_params = list(ansatz.parameters)

    qnn = EstimatorQNN(
        estimator=estimator,
        circuit=vqc_circuit,
        input_params=input_params,
        weight_params=weight_params,
        observables=observable,
        input_gradients=True,
    )

    model = VQCModel(qnn)

    X_tr_t = torch.tensor(X_train, dtype=torch.float32)
    y_tr_t = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    X_te_t = torch.tensor(X_test, dtype=torch.float32)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()

    for ep in range(epochs):
        # shuffle
        perm = torch.randperm(len(X_tr_t))
        X_tr_t = X_tr_t[perm]
        y_tr_t = y_tr_t[perm]

        total_loss = 0.0
        model.train()
        for i in range(0, len(X_tr_t), batch_size):
            xb = X_tr_t[i:i+batch_size]
            yb = y_tr_t[i:i+batch_size]

            optimizer.zero_grad()
            preds = model(xb)
            loss  = loss_fn(preds, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {ep+1}/{epochs} - loss: {total_loss:.4f}")

    # Evaluation
    model.eval()
    with torch.no_grad():
        probs = model(X_te_t).numpy().flatten()
    y_pred = (probs > 0.5).astype(int)

    acc = accuracy_score(y_test, y_pred)
    print("\nTest Accuracy for", cfg["name"], ":", acc)
    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, digits=4))

    return acc, model, y_pred


# ============================================================
# 7. Run all 3 ansatz configurations & compare
# ============================================================
results = []
models = {}

for cfg in ansatz_configs:
    acc, mdl, y_pred = train_vqc(cfg, X_train_scaled, y_train, X_test_scaled, y_test)
    results.append((cfg["name"], acc))
    models[cfg["name"]] = mdl

# Sort by accuracy
results_sorted = sorted(results, key=lambda x: x[1], reverse=True)

print("\n" + "#"*80)
print("FINAL SUMMARY: VQC on German Credit (6 qubits, 3 ansatz)")
print("#"*80)
for name, acc in results_sorted:
    print(f"{name:20s} -> Accuracy: {acc:.4f}")
print("#"*80)

print("\nDone.")


Fetching German Credit dataset from OpenML...
Raw dataframe shape: (1000, 21)
Columns: ['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings_status', 'employment', 'installment_commitment', 'personal_status', 'other_parties', 'residence_since', 'property_magnitude', 'age', 'other_payment_plans', 'housing', 'existing_credits', 'job', 'num_dependents', 'own_telephone', 'foreign_worker', 'class']

Class distribution (full):
class
1    700
0    300
Name: count, dtype: int64

After one-hot encoding, shape: (1000, 48)
Initial split sizes: train = 700 , test = 300
Capped training set to 200 samples

After PCA, train shape: (200, 6) test shape: (300, 6)
Original encoded feature dim: 48  > qubits: 6 OK

Scaled PCA feature range (train): -3.0375974 to 3.1415927  (approx [-pi, pi])

Feature map: ZZFeatureMap with 6 qubits and reps=1.

Ansatz configurations:
  - ry_linear_reps1 : {'name': 'ry_linear_reps1', 'rotation_blocks': 'ry', 'entanglement_blocks': 'cx', 'ent

C:\Users\hisham.fawad\AppData\Local\Temp\ipykernel_42448\56052654.py:212: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  qnn = EstimatorQNN(


Epoch 1/3 - loss: 8.6855
Epoch 2/3 - loss: 8.5889
Epoch 3/3 - loss: 8.4482

Test Accuracy for ry_linear_reps1 : 0.7

Confusion matrix:
[[  0  90]
 [  0 210]]

Classification report:
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000        90
           1     0.7000    1.0000    0.8235       210

    accuracy                         0.7000       300
   macro avg     0.3500    0.5000    0.4118       300
weighted avg     0.4900    0.7000    0.5765       300


Training VQC with ansatz: ry_linear_reps2


c:\Users\hisham.fawad\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hisham.fawad\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hisham.fawad\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  

Epoch 1/3 - loss: 10.9322
Epoch 2/3 - loss: 10.3922
Epoch 3/3 - loss: 10.0128

Test Accuracy for ry_linear_reps2 : 0.3

Confusion matrix:
[[ 90   0]
 [210   0]]

Classification report:
              precision    recall  f1-score   support

           0     0.3000    1.0000    0.4615        90
           1     0.0000    0.0000    0.0000       210

    accuracy                         0.3000       300
   macro avg     0.1500    0.5000    0.2308       300
weighted avg     0.0900    0.3000    0.1385       300


Training VQC with ansatz: ry_full_reps1


c:\Users\hisham.fawad\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hisham.fawad\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hisham.fawad\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  

Epoch 1/3 - loss: 9.0281
Epoch 2/3 - loss: 8.7977
Epoch 3/3 - loss: 8.6184

Test Accuracy for ry_full_reps1 : 0.7

Confusion matrix:
[[  0  90]
 [  0 210]]

Classification report:
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000        90
           1     0.7000    1.0000    0.8235       210

    accuracy                         0.7000       300
   macro avg     0.3500    0.5000    0.4118       300
weighted avg     0.4900    0.7000    0.5765       300


################################################################################
FINAL SUMMARY: VQC on German Credit (6 qubits, 3 ansatz)
################################################################################
ry_linear_reps1      -> Accuracy: 0.7000
ry_full_reps1        -> Accuracy: 0.7000
ry_linear_reps2      -> Accuracy: 0.3000
################################################################################

Done.


c:\Users\hisham.fawad\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hisham.fawad\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hisham.fawad\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  

Improved VQC on German Credit (credit-g, OpenML)